<a href="https://colab.research.google.com/github/justorfc/Estadistica_Aplicada_con_Python_y_R/blob/main/9_Semana_9_Correlaci%C3%B3n_y_Regresi%C3%B3n_Lineal_Simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este es el Notebook de la propuesta estructurada para la **Semana 9**, que da inicio al **Eje IV: Regresión y Modelación Predictiva**. En esta semana, damos el salto de describir y modelar variables individuales a cuantificar matemáticamente cómo una variable afecta a otra.

### Semana 9: Correlación y Regresión Lineal Simple

**Resultado de aprendizaje:** Cuantifica e interpreta asociación y construye un modelo de regresión lineal simple, diferenciando correlación de causalidad y reconociendo los límites de predicción.

---

#### Sesión 1: Asociación, Causalidad y el Método de Mínimos Cuadrados (80 - 90 minutos)

**Objetivo:** Comprender conceptualmente la diferencia entre asociación estadística y causalidad física, y ajustar un modelo de regresión lineal simple interpretando sus coeficientes.

* **20 min - Diálogo socrático y cálculo manuscrito (Lápiz y papel):**
* *Situación:* Un agricultor nota que a mayor lámina de riego, mayor rendimiento de su cultivo. ¿Podemos predecir cuánto producirá si aplicamos exactamente 450 mm de agua?
* *Actividad:* Los estudiantes dibujan un plano cartesiano (X = Riego, Y = Rendimiento) con 5 puntos simulados. A mano alzada, trazan la línea que consideran se ajusta mejor (línea de mejor ajuste). Se introduce el concepto matemático del intercepto (¿cuánto produce si riego = 0?) y la pendiente (¿cuánto aumenta el rendimiento por cada milímetro extra de agua?).


* **45 min - Exploración en Google Colab (Python):**
* Carga del cuaderno de la semana 9.
* Cálculo del coeficiente de Correlación de Pearson.
* Uso de la librería `statsmodels` para ajustar un modelo de regresión mediante Mínimos Cuadrados Ordinarios (OLS).
* Interpretación de la salida estadística: Coeficientes, valor $p$ y el coeficiente de determinación ($R^2$).


* **15 min - Reflexión manuscrita:**
* Peligros de la extrapolación: ¿Qué pasa matemáticamente (y agronómicamente) si predicimos el rendimiento para 2000 mm de agua basándonos en una línea recta?



---

#### Sesión 2: Predicción, Extrapolación y Transición a R (80 - 90 minutos)

**Objetivo:** Trasladar la modelación paramétrica al entorno de R, utilizando la sintaxis de fórmulas estadísticas estándar `y ~ x`.

* **20 min - La fórmula estadística en R:**
* Explicación de cómo R (y `statsmodels` en Python) utiliza la notación de tilde `~` para expresar "Y depende de X".


* **25 min - Prompts para modelación lineal en R:**
* Demostración de cómo instruir a la IA para replicar la regresión usando la función nativa `lm()` de R y extraer la tabla resumen con `summary()`.


* **40 min - Reto en Posit Cloud:**
* Los estudiantes ejecutan el flujo en su documento RMarkdown. Se les pide que agreguen una línea de tendencia a un gráfico de dispersión usando `ggplot2` y `geom_smooth(method = "lm")`, registrando todo en la bitácora de IA.



---

A continuación, el contenido para las celdas de tu cuaderno de Google Colab.

---

### Celda de Texto 1


# Semana 9: Correlación y Regresión Lineal Simple
**Asignatura:** Estadística Aplicada con Python y R  
**Programa:** Ingeniería Agrícola - Universidad de Sucre  
**Profesor:** Justo Rafael Fuentes Cuello  

---

### Situación de Interés: Prediciendo el Rendimiento
En agronomía y gestión del agua, muchas veces queremos predecir una variable difícil o costosa de medir a partir de una más sencilla.
Imagina que estamos evaluando el efecto de la **Lámina de Riego (mm)** sobre el **Rendimiento de maíz (ton/ha)**. Sabemos empíricamente que a más agua, más producción (hasta cierto punto).

Hoy pasaremos de decir "están relacionadas" a cuantificar **exactamente** cuánto aumenta el rendimiento por cada milímetro extra de agua. Para esto, usaremos la Correlación de Pearson y la Regresión Lineal Simple.

```

### Celda de Código 1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf # Librería para modelación estadística clásica

sns.set_theme(style="whitegrid")
np.random.seed(2026)

# Simulamos datos de 40 parcelas experimentales
# Lámina de riego aplicada durante el ciclo (entre 200 y 600 mm)
riego_mm = np.random.uniform(200, 600, 40)

# El rendimiento base es de 2 ton/ha, más 0.015 ton por cada mm de agua, más un error aleatorio
error = np.random.normal(loc=0, scale=0.8, size=40)
rendimiento_ton = 2.0 + (0.015 * riego_mm) + error

df_cultivo = pd.DataFrame({
    'Riego_mm': riego_mm,
    'Rendimiento_ton_ha': rendimiento_ton
})

print("Datos experimentales cargados.")
df_cultivo.head()

### 1. Cuantificando la Asociación: Correlación de Pearson
Primero, verificamos visualmente la relación con un diagrama de dispersión y calculamos el Coeficiente de Correlación de Pearson ($r$).
Este valor oscila entre -1 y 1. Un valor cercano a 1 indica una fuerte asociación lineal positiva.

```

### Celda de Código 2

In [ ]:
# Gráfico de dispersión básico
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df_cultivo, x='Riego_mm', y='Rendimiento_ton_ha', color='blue', s=60)
plt.title('Dispersión: Lámina de Riego vs Rendimiento')
plt.xlabel('Lámina de Riego (mm)')
plt.ylabel('Rendimiento (ton/ha)')
plt.show()

# Cálculo de la Correlación
correlacion = df_cultivo['Riego_mm'].corr(df_cultivo['Rendimiento_ton_ha'])
print(f"Coeficiente de Correlación de Pearson (r): {correlacion:.3f}")

### 2. El Modelo de Regresión Lineal (OLS)
La correlación nos dice que están asociados, pero no nos permite predecir.
Para predecir, necesitamos encontrar la ecuación de la línea recta ($Y = \beta_0 + \beta_1X$) que pase lo más cerca posible de todos los puntos. Esto se logra minimizando el error cuadrado (Ordinary Least Squares - OLS).

En Python, usaremos la notación de fórmulas estadísticas de `statsmodels`: `Y ~ X`.

```

### Celda de Código 3

In [ ]:
# Ajustamos el modelo: Rendimiento depende de (~) Riego
modelo_ols = smf.ols(formula='Rendimiento_ton_ha ~ Riego_mm', data=df_cultivo).fit()

# Mostramos el resumen estadístico formal
print(modelo_ols.summary())

# CONSULTA CON EL AGENTE DE IA, QUE SON CADA UNO DE LOS PARAMETROS Y COMO SE INTERPRETAN

### 3. Interpretación y Predicción
Del cuadro estadístico anterior, los números más importantes para un ingeniero son:
*   **Intercepto (Intercept):** El rendimiento teórico si el riego fuera 0 mm.
*   **Pendiente (Riego_mm):** El aumento en toneladas por cada 1 mm adicional de riego.
*   **R-squared ($R^2$):** Qué porcentaje de la variabilidad del rendimiento es explicada exclusivamente por el agua aplicada.

Ahora, ¡vamos a predecir! ¿Qué rendimiento obtendríamos si aplicamos 450 mm?

```

### Celda de Código 4

In [ ]:
# Extraemos los coeficientes
intercepto = modelo_ols.params['Intercept']
pendiente = modelo_ols.params['Riego_mm']
r_cuadrado = modelo_ols.rsquared

print(f"Ecuación del Modelo: Rendimiento = {intercepto:.3f} + ({pendiente:.4f} * Riego)")
print(f"La lámina de riego explica el {r_cuadrado*100:.1f}% de la variabilidad del rendimiento.\n")

# Predicción para un valor específico (450 mm)
nuevo_dato = pd.DataFrame({'Riego_mm': [450]})
prediccion = modelo_ols.predict(nuevo_dato)
print(f"Para un riego de 450 mm, el rendimiento esperado es: {prediccion[0]:.2f} ton/ha")

# Visualización de la línea de mejor ajuste
plt.figure(figsize=(7, 5))
sns.regplot(data=df_cultivo, x='Riego_mm', y='Rendimiento_ton_ha', color='blue', scatter_kws={'s':50}, line_kws={'color':'red'})
plt.plot(450, prediccion[0], marker='*', color='gold', markersize=15, markeredgecolor='black', label='Predicción (450mm)')
plt.title('Modelo de Regresión Lineal Simple ajustado')
plt.xlabel('Lámina de Riego (mm)')
plt.ylabel('Rendimiento (ton/ha)')
plt.legend()
plt.show()

### 🛑 Reflexión y Reserva Cognitiva (Síntesis manuscrita)
Toma tu lápiz y cuaderno, y analiza estos límites de la regresión:
1. Revisa el valor del intercepto generado por la máquina. ¿Tiene sentido agronómico ese número si asumimos que a una planta de maíz le aplicamos 0 mm de agua durante todo su ciclo?
2. **El peligro de la extrapolación:** Si aplicamos la ecuación para predecir el rendimiento con una inundación de 3000 mm de agua, la recta dirá que el rendimiento será altísimo. ¿Qué pasa en la realidad física de un cultivo si se le aplican 3000 mm de agua? ¿Por qué la línea recta matemática se equivoca aquí?
3. ¿Por qué el hecho de que el riego y el rendimiento estén altamente correlacionados (asociación) no demuestra por sí solo una "causa" inequívoca? (Pista: ¿qué pasaría si en las parcelas que más regamos también coincidió que el suelo era más fértil?).

---

### Instrucciones para el reto en R (Trabajo Autónomo y Sesión 2)

**Misión:** La sintaxis de fórmula `Y ~ X` que acabas de ver en `statsmodels` fue inventada originalmente en el lenguaje S, el antecesor de **R**. Tu reto es realizar este modelo paramétrico en **Posit Cloud**.

**Pasos a seguir:**
1. Abre tu proyecto en Posit Cloud y genera un nuevo documento RMarkdown.
2. Consulta a tu asistente de IA (ChatGPT, Gemini, etc.) usando este *prompt*:
   > *"Actúa como profesor de estadística con R. En Python ajustamos un modelo de regresión lineal (OLS) para predecir el Rendimiento en función del Riego usando la librería statsmodels y la fórmula 'Rendimiento ~ Riego'. Necesito replicar esto en R. Genera código para simular un dataset similar, y muéstrame cómo usar la función base `lm()` para calcular el modelo. Luego, muéstrame cómo ver el resumen estadístico con `summary()`, cómo hacer una predicción para 450 mm con `predict()`, y cómo graficar la recta sobre los puntos usando `ggplot2` y `geom_smooth(method = 'lm')`. Explícalo para incluirlo en un documento RMarkdown."*
3. Observa la salida de `summary(modelo)` en R. Busca los "Estimate" (estimaciones de intercepto y pendiente) y el "Multiple R-squared". Son idénticos matemáticamente a los de Python.
4. **Entrega:** Renderiza tu RMarkdown. En la sección final, incluye tu "Bitácora de IA", comentando sobre las similitudes entre la salida estadística de R y la de `statsmodels` en Python, y reportando cualquier ajuste de código que tuviste que hacer.